# Demo: CADIP requests with temporal filters

See https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-834   
This demo shows the usage of advanced temporal filters in rs-server-cadip, such as ValCover (using t_contains) or ValIntersect (using t_intersects).

## 0 - Initialization

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *

init_demo()

# Reload the global vars again
from resources.utils import *  

## 1 - Full CADIP data

Here is the list of CADIP items we will run requests on. There are 4 items in the collection "mti_sentinel1" that we are going to use.

In [ ]:
search_params = {
    "owner_id": OWNER_ID,
    "max_items": 100,
    "collections": ["mti_sentinel1"],
    "stac_filter": {}
}
cadip_items_full_list = cadip_client.search(**search_params)
assert len(cadip_items_full_list)==4

display(cadip_items_full_list)

## 2 - Filter definitions

We are creating 3 filters matching 3 tests cases:
  - TC1: ValCover filter retrieving item 1
  - TC2: ValIntersect filter on one field retrieving item 2
  - TC3: ValIntersect filter retrieving 2 items

In [ ]:
# TC1: Values matching item with ID S1A_20231120061537234567
tc1_filter = {
    "op": "t_contains",
    "args": [
        {"interval": [{"property": "cadip:planned_data_start"}, {"property": "cadip:planned_data_stop"}]},
        {"interval": ["2023-11-20T06:06:37.234Z", "2023-11-20T06:14:37.234000Z"]}
    ]
}

# TC2: Interval in which we can find the "datetime" value of item with ID S1A_20220715090550123456
tc2_filter = {
    "op": "t_intersects",
    "args": [
        {"property": "datetime"},
        {"interval": ["2022-07-14T09:00:00.000000Z", "2022-07-16T09:00:00.000000Z"]}
    ]
}

# TC3: Interval with two properties in which we can find items with IDs S1A_20210410031928012345 and S1A_20200105072204051312
tc3_filter = {
    "op": "t_intersects",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "datetime"}]},
        {"interval": ["2020-01-01T07:00:00.000Z", "2021-06-01T07:00:00.000Z"]}
    ]
}

## 3 - Tests

In [ ]:
# TEST CASE 1
search_params = {
    "owner_id": OWNER_ID,
    "max_items": 100,
    "collections": ["mti_sentinel1"],
    "stac_filter": tc1_filter
}
retrieved_cadip_items = cadip_client.search(**search_params)
assert len(retrieved_cadip_items)==1
assert retrieved_cadip_items[0].id == "S1A_20231120061537234567"

display(retrieved_cadip_items)

In [ ]:
# TEST CASE 2
search_params = {
    "owner_id": OWNER_ID,
    "max_items": 100,
    "collections": ["mti_sentinel1"],
    "stac_filter": tc2_filter
}
retrieved_cadip_items = cadip_client.search(**search_params)
assert len(retrieved_cadip_items)==1
assert retrieved_cadip_items[0].id == "S1A_20220715090550123456"

display(retrieved_cadip_items)

In [ ]:
# TEST CASE 3
search_params = {
    "owner_id": OWNER_ID,
    "max_items": 100,
    "collections": ["mti_sentinel1"],
    "stac_filter": tc3_filter
}
retrieved_cadip_items = cadip_client.search(**search_params)
assert len(retrieved_cadip_items)==2
assert retrieved_cadip_items[0].id == "S1A_20210410031928012345"
assert retrieved_cadip_items[1].id == "S1A_20200105072204051312"

display(retrieved_cadip_items)